# ✅ SÍNTESE 3 — FASE DE VALIDAÇÃO
## *Notebooks 16 a 21 | Frentes 6 a 11: Provando que funciona no mundo real*

---

> **💡 Para qualquer leitor:** Até agora, construímos e otimizamos o sistema. Nesta fase, **colocamos ele à prova** com condições cada vez mais difíceis: dados que ele nunca viu, ambientes barulhentos, e um júri independente (protocolo DCASE 2025).

---

## 📋 CRITÉRIOS DE ACEITAÇÃO (imutáveis desde o início)

| Critério | Limite | Status nesta fase |
|----------|--------|-------------------|
| **pAUC@0.1 (Score DCASE)** | > 0.80 | 🎯 Meta: superar 0.90 |
| **Latência de Inferência** | < 50 ms | 🎯 Meta: < 20ms total |
| **Memória Usada** | < 4 MB | 🎯 Meta: < 1MB total |

---

## 🗂️ O QUE ESTA SÍNTESE COBRE

```
Notebook 16 → Frente 6:  Threshold Gamma + FPR Alvo (controle formal do limiar)
Notebook 17 → Frente 7:  Outlier Exposure Real via DCASE (aprender de anomalias externas)
Notebook 18 → Frente 8:  Baselines Não Supervisionados (arena comparativa final)
Notebook 19 → Frente 9:  Pré-treino e Augmentation (Tiny-AST como auxiliar)
Notebook 20 → Frente 10: Ajustes XAI (explicabilidade visual e tabular)
Notebook 21 → Frente 11: Consolidação Multi-Critério Final
```


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import gamma as gamma_dist
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor'] = '#161b22'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['axes.edgecolor'] = '#30363d'
plt.rcParams['grid.color'] = '#30363d'

print('✅ Fase 3: Validação. Hora de provar que funciona!')

---
# 📖 NOTEBOOK 16 — Frente 6: Threshold Gamma + FPR Alvo
### *O limiar matemático que controla a sensibilidade do alarme*

## 🧒 Explicação simples

Voltando ao exemplo do detector de fumaça: **como você regula a sensibilidade?**

- Muito sensível → alarma em qualquer cheiro de café (**falso positivo**)
- Pouco sensível → não detecta um incêndio no início (**falso negativo**)

Na Frente 1, introduzimos a ideia do Limiar Gamma. Aqui, **formalizamos matematicamente** o controle:

```
Dado um FPR alvo (ex: 5%)
→ Ajuste a curva Gamma aos erros de sons normais
→ O limiar é o percentil (1 - FPR) da curva
→ Ex: FPR=5% → limiar = percentil 95% da Gamma
```

## 🔬 Por que Distribuição Gamma?

Os erros de reconstrução (MSE, NMF error) são:
- **Sempre positivos** (não existem erros negativos)
- **Assimétricos** (muitos erros pequenos, poucos grandes)
- Matematicamente, seguem distribuição Gamma — validado por teste KS (Kolmogorov-Smirnov)

## 📊 Controle do FPR com Limiar Gamma

| FPR Alvo | Limiar Gamma | Sensibilidade | pAUC@0.1 |
|----------|-------------|--------------|----------|
| 1% | 0.89 | Muito alto | 0.95 |
| **5%** | **0.74** | **Ótimo (industrial)** | **0.94** |
| 10% | 0.61 | Moderado | 0.91 |
| 20% | 0.45 | Baixo | 0.85 |

## 🟢 DECISÃO: FPR ALVO = 5% como padrão industrial

> Escolhido FPR alvo = 5%: equilíbrio entre não perder falhas reais e não gerar alarmes excessivos.
> 
> O limiar agora é **calculado automaticamente** a cada novo lote de dados — não é mais um número fixo escolhido manualmente.

## 🔴 DESCARTADO

> **Descartado:** Threshold fixo universal (ex: MSE > 0.5 → anomalia)
> 
> **Razão:** O mesmo threshold fixo que funciona bem num sensor industrial pode gerar 50% de falsos positivos num celular diferente. O Gamma ajusta-se automaticamente ao sensor usado.


In [ ]:
# ============================================================
# VISUALIZAÇÃO — Limiar Gamma controlando o FPR
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

# Distribuição Gamma com FPR alvo
ax1 = axes[0]
ax1.set_facecolor('#161b22')
np.random.seed(42)
x_range = np.linspace(0, 3, 1000)

# Curva Gamma ajustada a erros normais
a, scale = 2.5, 0.3
gamma_pdf = gamma_dist.pdf(x_range, a=a, scale=scale)
ax1.fill_between(x_range, gamma_pdf, alpha=0.3, color='#2ea043', label='Distribuição dos Erros Normais')
ax1.plot(x_range, gamma_pdf, color='#2ea043', linewidth=2.5)

# Diferentes limiares
for fpr_target, color, label in [
    (0.01, '#da3633', 'FPR 1% (muito rigoroso)'),
    (0.05, '#f0883e', 'FPR 5% ✅ (escolhido)'),
    (0.10, '#9e6a03', 'FPR 10% (moderado)'),
]:
    threshold = gamma_dist.ppf(1 - fpr_target, a=a, scale=scale)
    ax1.axvline(threshold, color=color, linewidth=2, linestyle='--', label=f'{label}: T={threshold:.2f}')
    ax1.fill_between(x_range[x_range >= threshold], gamma_pdf[x_range >= threshold], 
                     alpha=0.4, color=color)

ax1.set_title('🎯 Limiar Gamma: Controlando FPR com Precisão', color='white', fontsize=11)
ax1.set_xlabel('Erro de Reconstrução', color='white')
ax1.set_ylabel('Densidade de Probabilidade', color='white')
ax1.legend(frameon=False, labelcolor='white', fontsize=8)
ax1.set_xlim(0, 3)

# Comparação FPR alvo × pAUC
ax2 = axes[1]
ax2.set_facecolor('#161b22')
fpr_targets = [0.01, 0.05, 0.10, 0.20]
pauc_vals = [0.95, 0.94, 0.91, 0.85]
fprs_achieved = [0.012, 0.048, 0.098, 0.19]

ax2_twin = ax2.twinx()
line1, = ax2.plot(fpr_targets, pauc_vals, color='#2ea043', marker='o', linewidth=2.5, markersize=10, label='pAUC@0.1')
line2, = ax2_twin.plot(fpr_targets, fprs_achieved, color='#1f6feb', marker='s', linewidth=2.5, markersize=10, label='FPR Real')
ax2.axvline(0.05, color='#f0883e', linestyle='--', linewidth=2, label='Escolhido: 5%')
ax2.set_xlabel('FPR Alvo', color='white')
ax2.set_ylabel('pAUC@0.1', color='#2ea043')
ax2_twin.set_ylabel('FPR Real Alcançado', color='#1f6feb')
ax2_twin.yaxis.label.set_color('#1f6feb')
ax2_twin.tick_params(axis='y', colors='#1f6feb')
ax2.set_title('⚖️ Balanço: FPR alvo × pAUC obtido', color='white', fontsize=11)
ax2.legend([line1, line2], ['pAUC@0.1', 'FPR Real'], frameon=False, labelcolor='white', fontsize=9)

plt.tight_layout()
plt.show()

---
# 📖 NOTEBOOK 17 — Frente 7: Outlier Exposure Real via DCASE
### *Usando anomalias externas para endurecer as fronteiras*

## 🧒 Explicação simples

Imagine um professor que, além de ensinar com o livro, traz exemplos reais de provas antigas de outras escolas. O aluno fica muito melhor preparado!

**Outlier Exposure (OE)** funciona assim: durante o treino, mostramos ao modelo anomalias reais de **outras máquinas** (do dataset DCASE), para que ele aprenda a forma geral de uma anomalia, não apenas a anomalia específica da nossa máquina.

## 📊 Impacto do Outlier Exposure

| Configuração | pAUC@0.1 (próprio) | pAUC@0.1 (cross-domain) | Generalização |
|-------------|--------------------|-----------------------|--------------|
| Sem OE | 0.94 | 0.68 | Fraca |
| OE parcial (10% dados ext.) | 0.93 | 0.72 | Moderada |
| **OE completo (DCASE externo)** | **0.94** | **0.76** | **Boa** |
| OE excessivo (50% dados ext.) | 0.90 | 0.74 | Risco de bias |

## ⚠️ DECISÃO: ESTRATÉGIA AUXILIAR (opcional)

> O OE melhora cross-domain de 0.68 para 0.76, mas adiciona complexidade no setup do treinamento.
>
> **Decisão:** OE é útil como **estratégia de robustez adicional** quando se tem acesso ao dataset DCASE. Não é obrigatório na pipeline mínima.
>
> **Esta frente também consolidou a separação formal entre:**
> - **Campeão Supervisionado:** XGBoost + HHT+UKF + Super-Vetor
> - **Campeão Não-Supervisionado:** GMM + Limiar Gamma
> 
> E alinhamento ao protocolo DCASE 2025 Task 2 (First-Shot Unsupervised AAD).


---
# 📖 NOTEBOOK 18 — Frente 8: Baselines Não Supervisionados
### *Arena final: quem vence no regime sem rótulos?*

## 🧒 Explicação simples

Na fábrica, muitas vezes não temos exemplos de falhas para treinar — o sistema precisa detectar anomalias **sem nunca ter visto uma**. Testamos 3 abordagens:

| Detetive | Estratégia | Analogia |
|----------|-----------|----------|
| **Mahalanobis** | Distância da "nuvem" de dados normais | Sabe como é a casa normal, rejeita tudo muito diferente |
| **GMM** | Múltiplos modos de normalidade | Sabe que a casa muda conforme o dia da semana |
| Isolation Forest | Tenta isolar pontos raros | Separa quem fica sozinho na festa |

## 📊 Arena Comparativa — Regime Não-Supervisionado

| Modelo | pAUC@0.1 | Latência | Memória | Comportamento em Domain Shift |
|--------|----------|----------|---------|-------------------------------|
| **GMM (k=5)** | **0.88** | **20ms** | **0.8MB** | **Robusto (multi-modal)** |
| **Mahalanobis** | **0.82** | **5ms** | **0.2MB** | **Aceitável (mono-modal)** |
| Isolation Forest | 0.71 | 35ms | 2.1MB | Fraco em multi-source |
| LSTM-AE | 0.79 | 120ms | 8MB | Viola limites Edge |

## 🔴 DESCARTADOS NO REGIME NÃO-SUPERVISIONADO

> **Isolation Forest:** pAUC 0.71 (abaixo do limite 0.80). Não detecta bem anomalias em espaços de alta dimensão.
>
> **LSTM-AE:** 120ms e 8MB — viola latência e memória. O conceito (erro de reconstrução) é mantido via NMF leve.

## 🟢 ARQUITETURA DUAL CONFIRMADA

```
Regime A (Supervisionado):     XGBoost  → pAUC 0.94, 12ms, <1MB
Regime B (Não-Supervisionado): GMM+Gamma → pAUC 0.88, 20ms, 0.8MB
Fallback Não-Sup:         Mahalanobis  → pAUC 0.82,  5ms, 0.2MB
```

> **Descoberta crítica:** Não existe um único modelo universal. O regime depende se você tem rótulos de anomalia disponíveis. O sistema oferece ambas as opções.


In [ ]:
# ============================================================
# VISUALIZAÇÃO — Arena Comparativa Multi-critério
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('⚔️ Arena Comparativa: Todos os Modelos — 3 Critérios Inegociáveis', 
             color='white', fontsize=13, fontweight='bold')

modelos = ['GMM\n(k=5)', 'Mahalanobis', 'Isolation\nForest', 'LSTM-AE', 'XGBoost\n+HHT']
pauc_vals = [0.88, 0.82, 0.71, 0.79, 0.94]
latency_vals = [20, 5, 35, 120, 12]
memory_vals = [0.8, 0.2, 2.1, 8.0, 0.7]
colors = ['#2ea043', '#1f6feb', '#da3633', '#da3633', '#9e6a03']
regimes = ['Não-Sup', 'Não-Sup\n(fallback)', 'Descartado', 'Descartado', 'Supervisionado']

# pAUC
ax = axes[0]
ax.set_facecolor('#161b22')
bars = ax.bar(modelos, pauc_vals, color=colors, alpha=0.85)
ax.axhline(0.80, color='#f0883e', linestyle='--', linewidth=2, label='Limite mínimo')
ax.set_title('pAUC@0.1 (Score DCASE)', color='white', fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend(frameon=False, labelcolor='white', fontsize=9)
for bar, val in zip(bars, pauc_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.2f}', 
            ha='center', color='white', fontsize=9, fontweight='bold')

# Latência
ax = axes[1]
ax.set_facecolor('#161b22')
bars = ax.bar(modelos, latency_vals, color=colors, alpha=0.85)
ax.axhline(50, color='#f0883e', linestyle='--', linewidth=2, label='Limite máximo (50ms)')
ax.set_title('Latência de Inferência (ms)', color='white', fontsize=11)
ax.legend(frameon=False, labelcolor='white', fontsize=9)
for bar, val in zip(bars, latency_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 2, f'{val}ms', 
            ha='center', color='white', fontsize=9, fontweight='bold')

# Memória
ax = axes[2]
ax.set_facecolor('#161b22')
bars = ax.bar(modelos, memory_vals, color=colors, alpha=0.85)
ax.axhline(4.0, color='#f0883e', linestyle='--', linewidth=2, label='Limite máximo (4MB)')
ax.set_title('Memória do Modelo (MB)', color='white', fontsize=11)
ax.legend(frameon=False, labelcolor='white', fontsize=9)
for bar, val in zip(bars, memory_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.1, f'{val}MB', 
            ha='center', color='white', fontsize=9, fontweight='bold')

# Legenda de cores
legend_handles = [
    mpatches.Patch(color='#9e6a03', label='XGBoost (Supervisionado ✅)'),
    mpatches.Patch(color='#2ea043', label='GMM (Não-Sup ✅)'),
    mpatches.Patch(color='#1f6feb', label='Mahalanobis (Fallback ✅)'),
    mpatches.Patch(color='#da3633', label='Descartado 🔴'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=4, frameon=False, 
           fontsize=10, labelcolor='white', bbox_to_anchor=(0.5, -0.05))

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

---
# 📖 NOTEBOOK 19 — Frente 9: Pré-treino e Augmentation
### *O Tiny-AST: o gigante útil como ajudante*

## 🧒 Explicação simples

Imagine contratar um especialista renomado para dar consultoria. Ele sabe muito, mas é caro e lento. Em vez de deixá-lo fazer tudo, usamos sua expertise apenas para criar um "relatório especial" que um assistente mais rápido vai usar.

O **Tiny-AST** (Audio Spectrogram Transformer com ~1M parâmetros) é esse especialista:
- Entende profundamente padrões acústicos complexos
- Mas é lento demais para Edge (85ms + 12MB)
- **Solução:** Usamos suas representações internas (embeddings) apenas para enriquecer o XAI e validar o diagnóstico

## 📊 Tiny-AST como auxiliar vs como motor

| Uso | pAUC@0.1 | Latência | Memória | Viável Edge? |
|-----|----------|----------|---------|-------------|
| Motor principal | 0.91 | 85ms | 12MB | 🔴 Não |
| **Embeddings → XAI** | **+2% validação** | **+0ms inferência** | **+0MB inferência** | **✅ Sim** |

> Os embeddings do Tiny-AST são calculados **uma vez no Cloud** e usados para validar interpretações XAI. O dispositivo de borda nunca carrega o Tiny-AST.

## 🔴 POR QUE O TINY-AST FOI DESCARTADO COMO MOTOR?

> **Critério violado: Latência (85ms) e Memória (12MB)**
>
> Em testes com dados escassos (poucas anomalias disponíveis), o Tiny-AST também tendeu a memorizar características do sensor — o mesmo problema que o CNN.
>
> **Regra confirmada:** Em AAD com dados escassos, modelos leves + boa engenharia de features superam Transformers pesados.


---
# 📖 NOTEBOOK 20 — Frente 10: Ajustes XAI
### *Tornando o sistema explicável: por que o alarme tocou?*

## 🧒 Explicação simples

Um alarme que toca sem explicação é inútil para um técnico de manutenção. Ele precisa saber:
- **Quando** a anomalia foi detectada (timestamp)
- **Onde no espectro** de frequências está a anomalia
- **Qual tipo** de falha (impacto, atrito, vibração)

O **XAI** (Explainable AI) transforma o score de anomalia em evidência física compreensível.

## 🔬 O que o XAI gera?

**1. Mapa de Saliência (Heatmap):** Qual parte do espectrograma ativou o alarme?

**2. Hotspots Temporais:**

| Hotspot | Tempo (s) | Intensidade | Diagnóstico |
|---------|-----------|-------------|-------------|
| H1 | 0.614 | 5.41 | Impacto principal (gatilho da anomalia) |
| H2 | 1.126 | 4.99 | Harmônico estrutural da mesma falha |
| H3 | 0.666 | 4.75 | Eco temporal (recorrência do mecanismo) |

**3. Análise por Bandas de Frequência:**

| Faixa | Frequência (Hz) | Ativação Média | Diagnóstico Físico |
|-------|----------------|----------------|--------------------|
| Baixa | 0 – 800 | 0.0258 | Fenômenos mecânicos (atrito, impacto) |
| Média | 800 – 2500 | 0.0177 | Harmônicos estruturais |
| Alta | 2500 – 5000 | 0.0031 | Ruído elétrico/eletrônico |

## 🟢 DECISÃO: XAI É COMPONENTE ESSENCIAL

> O XAI não melhora o pAUC, mas transforma o sistema de um **caixa-preta** em um **sistema de diagnóstico**.
>
> Para um técnico de manutenção, saber que a anomalia está em 0-800 Hz (faixa de atrito mecânico) é muito mais útil que um score de 0.87.

## 🔴 DESCARTADOS NESTE NOTEBOOK

> **Descartado:** Modelos black-box sem pathway de explicabilidade
>
> **Descartado:** Score numérico sem contexto físico


In [ ]:
# ============================================================
# VISUALIZAÇÃO — XAI: Heatmap + Análise de Bandas
# ============================================================
fig = plt.figure(figsize=(16, 9))
fig.patch.set_facecolor('#0d1117')
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.35)

# Heatmap de Saliência
ax1 = fig.add_subplot(gs[:, 0])
ax1.set_facecolor('#161b22')
np.random.seed(42)
heatmap = np.random.rand(64, 50) * 0.1
# Região de anomalia H1
heatmap[15:35, 28:36] += 0.8
# Região H2
heatmap[10:25, 44:50] += 0.5
# Região H3
heatmap[15:30, 30:35] += 0.35

im = ax1.imshow(heatmap, aspect='auto', origin='lower', cmap='hot', interpolation='bilinear')
ax1.set_title('🔥 Mapa de Saliência XAI\n(onde o alarme foi ativado)', color='white', fontsize=10)
ax1.set_xlabel('Tempo (janelas)', color='white', fontsize=9)
ax1.set_ylabel('Frequência (mel-bins)', color='white', fontsize=9)
plt.colorbar(im, ax=ax1, label='Ativação', shrink=0.8)
ax1.annotate('H1 (principal)', xy=(32, 25), xytext=(36, 45),
             arrowprops=dict(arrowstyle='->', color='white', lw=2),
             color='white', fontsize=9, fontweight='bold')
ax1.annotate('H2', xy=(47, 17), xytext=(40, 5),
             arrowprops=dict(arrowstyle='->', color='#f0883e', lw=1.5),
             color='#f0883e', fontsize=9)

# Hotspots Temporais
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor('#161b22')
hotspots = {'H1': (0.614, 5.41), 'H2': (1.126, 4.99), 'H3': (0.666, 4.75)}
colors_h = ['#da3633', '#f0883e', '#9e6a03']
for (name, (t, v)), color in zip(hotspots.items(), colors_h):
    ax2.scatter(t, v, s=200, color=color, zorder=5, label=f'{name}: t={t}s, I={v:.2f}')
    ax2.annotate(name, xy=(t, v), xytext=(t+0.05, v+0.05), color=color, fontsize=11, fontweight='bold')
ax2.set_title('⏱️ Hotspots Temporais', color='white', fontsize=10)
ax2.set_xlabel('Tempo (s)', color='white', fontsize=9)
ax2.set_ylabel('Intensidade de Ativação', color='white', fontsize=9)
ax2.legend(frameon=False, labelcolor='white', fontsize=8)
ax2.set_xlim(0, 1.3); ax2.set_ylim(4.5, 5.6)

# Análise por Bandas
ax3 = fig.add_subplot(gs[1, 1])
ax3.set_facecolor('#161b22')
bandas = ['Baixa\n(0-800 Hz)', 'Média\n(800-2500 Hz)', 'Alta\n(2500-5000 Hz)']
ativacoes = [0.0258, 0.0177, 0.0031]
band_colors = ['#da3633', '#9e6a03', '#8b949e']
bars = ax3.bar(bandas, ativacoes, color=band_colors, alpha=0.85)
ax3.set_title('📊 Ativação por Banda de Frequência', color='white', fontsize=10)
ax3.set_ylabel('Ativação Média', color='white', fontsize=9)
for bar, val in zip(bars, ativacoes):
    ax3.text(bar.get_x() + bar.get_width()/2, val + 0.0005, f'{val:.4f}',
             ha='center', color='white', fontsize=10, fontweight='bold')

# Diagnóstico integrado
ax4 = fig.add_subplot(gs[:, 2])
ax4.set_facecolor('#161b22')
ax4.axis('off')
diag_text = (
    "🔍 DIAGNÓSTICO XAI INTEGRADO\n"
    "════════════════════════════\n\n"
    "ANOMALIA DETECTADA: ✅ SIM\n"
    "Score DCASE: 0.94\n"
    "FPR estimado: < 5%\n\n"
    "━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
    "QUANDO: t=0.614s (H1 principal)\n"
    "ONDE: Faixas Baixa + Média\n"
    "       (0–2500 Hz)\n"
    "TIPO: Impacto mecânico\n"
    "       com harmônicos estruturais\n\n"
    "━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
    "CAUSA PROVÁVEL:\n"
    "Desgaste ou impacto em\n"
    "componente rotativo\n"
    "(rolamento ou engrenagem)\n\n"
    "━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
    "AÇÃO RECOMENDADA:\n"
    "Inspeção prioritária\n"
    "no subsistema de transmissão"
)
ax4.text(0.05, 0.95, diag_text, transform=ax4.transAxes,
         color='white', fontsize=9.5, verticalalignment='top',
         fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#21262d', edgecolor='#2ea043', linewidth=2))

plt.suptitle('🧠 XAI Completo: Do Score ao Diagnóstico Físico Acionável', 
             color='white', fontsize=13, fontweight='bold')
plt.show()

---
# 📖 NOTEBOOK 21 — Frente 11: Consolidação Multi-Critério Final
### *A arena definitiva: tudo testado junto sob protocolo LOSO*

## 🧒 Explicação simples

Imagine a fase final de um campeonato: todos os times que sobreviveram às fases anteriores competem num único torneio com regras rígidas. Quem vencer é o campeão oficial.

Este notebook é essa final: todos os modelos sobreviventes competem sob **protocolo LOSO completo** com **4 fontes de dados** e **métrica DCASE oficial**.

## 📊 Ranking Final — Frente 11

| # | Modelo | Regime | pAUC@0.1 | Lat. | Mem. | DCASE-Ready? |
|---|--------|--------|----------|------|------|-------------|
| 1 | **XGBoost + HHT+UKF + Super-Vetor** | Supervisionado | **0.94** | **15ms** | **<1MB** | **✅ Sim** |
| 2 | **GMM (k=5) + Gamma** | Não-Supervisionado | **0.88** | **20ms** | **0.8MB** | **✅ Sim** |
| 3 | Mahalanobis + HHT | Não-Sup (fallback) | 0.82 | 8ms | 0.2MB | ✅ Sim |
| 4 | Tiny-AST + GMM | Misto | 0.91 | 92ms | 12.8MB | 🔴 Não |
| 5 | GRU + Mel | Supervisionado | 0.87 | 45ms | 3.2MB | ⚠️ Borderline |

## 🏆 CAMPEÕES OFICIAIS POR REGIME:

```
╔══════════════════════════════════════════════════════╗
║  PROTOCOLO A (Supervisionado):                       ║
║  XGBoost + HHT+UKF + Mixup + Super-Vetor            ║
║  pAUC: 0.94 | Lat: 15ms | Mem: <1MB                ║
╠══════════════════════════════════════════════════════╣
║  PROTOCOLO B (Não-Supervisionado):                   ║
║  GMM (k=5) + Gamma Threshold (FPR 5%)               ║
║  pAUC: 0.88 | Lat: 20ms | Mem: 0.8MB               ║
╚══════════════════════════════════════════════════════╝
```


---
# 🏆 RESUMO FINAL — FASE DE VALIDAÇÃO

## ✅ MANTIDOS E CONFIRMADOS:

| Frente | Técnica | Contribuição | Custo |
|--------|---------|-------------|-------|
| F6 | Limiar Gamma + FPR 5% | Controle formal do alarme | Zero |
| F7 | Outlier Exposure (DCASE) | +8% cross-domain | Só treino |
| F8 | GMM (k=5) como campeão não-sup | pAUC 0.88 | 20ms / 0.8MB |
| F8 | Mahalanobis como fallback | pAUC 0.82 | 5ms / 0.2MB |
| F9 | Tiny-AST como auxiliar XAI | Validação semântica | Zero em edge |
| F10 | XAI completo (heatmap+hotspot+banda) | Diagnóstico acionável | Zero |
| F11 | Arquitetura Dual (A + B) | Cobre ambos os regimes | - |

## 🔴 DESCARTADOS NESTA FASE:

| Técnica | Motivo | Critério |
|---------|--------|----------|
| Isolation Forest (não-sup) | pAUC 0.71 | Score DCASE |
| LSTM-AE (não-sup) | 120ms + 8MB | Latência + Memória |
| Tiny-AST como motor principal | 92ms + 12.8MB | Latência + Memória |
| GRU como motor principal | Inferior ao XGBoost em todos os critérios | Eficiência |
| Crença em um único modelo universal | Dois regimes necessários | Arquitetura |

---

## ➡️ PRÓXIMO PASSO: SINTESE_4_Confrontamento.ipynb
### *Notebooks 22-28: A decisão final — o modelo DIAMANTE CRISTAL*

---
*Síntese por: Emanoel Spanhol | Projeto AudioAlert | Pós-Graduação IA Aplicada — UniSENAI | 2025-2026*